# ⚔️ SmolLM2-1.7B Alignment Arena — Live Side-by-Side Demo

This notebook loads the **Base**, **SFT**, and **DPO** models and launches a live interactive **Gradio Arena** with real-time streaming responses!

### Models Compared:
1. 🔴 **Base Model (`HuggingFaceTB/SmolLM2-1.7B`):** Raw pre-trained foundation model.
2. 🟡 **SFT Model (`manojpaul9986/smollm2-1.7b-sft-lora`):** Supervised fine-tuned on SmolTalk.
3. 🟢 **DPO Model (`manojpaul9986/smollm2-1.7b-dpo-lora`):** Preference-aligned on UltraFeedback with step-by-step reasoning.

### Step 1: Install Dependencies

In [ ]:
!pip uninstall -y torchao
!pip install -q -U torch transformers peft trl gradio huggingface_hub accelerate


### Step 2: Download SFT & DPO Adapters from Hugging Face Hub

In [ ]:
import os
from huggingface_hub import snapshot_download

BASE_DIR = "/kaggle/working" if os.path.exists("/kaggle/working") else "./work"
SFT_DIR = os.path.join(BASE_DIR, "checkpoints/sft")
DPO_DIR = os.path.join(BASE_DIR, "checkpoints/dpo")

if not os.path.exists(SFT_DIR) or not os.listdir(SFT_DIR):
    print("Downloading SFT adapter from Hugging Face Hub...")
    snapshot_download(repo_id="manojpaul9986/smollm2-1.7b-sft-lora", local_dir=SFT_DIR)
else:
    print(f"Found existing SFT adapter at: {SFT_DIR}")

if not os.path.exists(DPO_DIR) or not os.listdir(DPO_DIR):
    print("Downloading DPO adapter from Hugging Face Hub...")
    snapshot_download(repo_id="manojpaul9986/smollm2-1.7b-dpo-lora", local_dir=DPO_DIR)
else:
    print(f"Found existing DPO adapter at: {DPO_DIR}")

print("✅ All adapters ready!")


### Step 3: Load Models into GPU Memory

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TextIteratorStreamer
from peft import PeftModel
from threading import Thread

MODEL_ID = "HuggingFaceTB/SmolLM2-1.7B"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")

# 1. Tokenizer
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token
if tokenizer.chat_template is None:
    tokenizer.chat_template = (
        "{% for message in messages %}"
        "{{'<|im_start|>' + message['role'] + '\n' + message['content'] + '<|im_end|>\n'}}"
        "{% endfor %}"
        "{% if add_generation_prompt %}{{'<|im_start|>assistant\n'}}{% endif %}"
    )

# 2. Load Base Model
print("Loading Base Model (SmolLM2-1.7B)...")
base_model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
base_model.eval()

# 3. Load SFT Model
print("Loading SFT Model...")
sft_base = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
sft_model = PeftModel.from_pretrained(sft_base, SFT_DIR).to(DEVICE)
sft_model.eval()

# 4. Load DPO Model (Mounted on top of merged SFT base)
print("Loading DPO Model...")
dpo_base_raw = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=torch.float16).to(DEVICE)
dpo_sft_merged = PeftModel.from_pretrained(dpo_base_raw, SFT_DIR).to(DEVICE).merge_and_unload()
dpo_model = PeftModel.from_pretrained(dpo_sft_merged, DPO_DIR).to(DEVICE)
dpo_model.eval()

print("🎉 All 3 models loaded successfully into GPU memory!")


### Step 4: Launch the Live Side-by-Side Gradio Arena

Run this cell to open the live interactive arena. Click the generated **public link** (or interact directly in the inline frame) to test prompts side-by-side!

In [ ]:
import gradio as gr
import time

def generate_single(model, prompt, chat, max_tokens, temperature, top_p):
    if chat:
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(messages, add_generation_prompt=True, return_tensors="pt").to(DEVICE)
    else:
        inputs = tokenizer(prompt, return_tensors="pt")["input_ids"].to(DEVICE)
        
    if inputs.ndim == 1:
        inputs = inputs.unsqueeze(0)
        
    with torch.no_grad():
        out = model.generate(
            input_ids=inputs,
            max_new_tokens=int(max_tokens),
            do_sample=temperature > 0.0,
            temperature=max(float(temperature), 0.01) if temperature > 0.0 else 1.0,
            top_p=float(top_p) if temperature > 0.0 else 1.0,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.encode("<|im_end|>")[0] if "<|im_end|>" in tokenizer.get_vocab() else tokenizer.eos_token_id
        )
    return tokenizer.decode(out[0][inputs.shape[1]:], skip_special_tokens=True).strip()

def arena_compare(prompt, max_tokens, temperature, top_p):
    # Sequential generation for clean memory management
    base_res = generate_single(base_model, prompt, False, max_tokens, temperature, top_p)
    sft_res = generate_single(sft_model, prompt, True, max_tokens, temperature, top_p)
    dpo_res = generate_single(dpo_model, prompt, True, max_tokens, temperature, top_p)
    return base_res, sft_res, dpo_res

custom_css = """
.gradio-container { font-family: 'Inter', sans-serif; }
.model-box textarea { font-size: 14px; line-height: 1.6; }
"""

with gr.Blocks(theme=gr.themes.Soft(primary_hue="indigo"), css=custom_css, title="SmolLM2 Alignment Arena") as demo:
    gr.Markdown("# ⚔️ SmolLM2-1.7B Post-Training Alignment Arena")
    gr.Markdown("Live side-by-side comparison across the full alignment lifecycle: **Base ➔ SFT ➔ DPO**.")
    
    with gr.Row():
        with gr.Column(scale=4):
            prompt_box = gr.Textbox(
                label="User Prompt",
                placeholder="Enter any reasoning, math, or instruction prompt...",
                lines=3,
                value="What is the difference between a software library and a framework? Explain with a simple analogy."
            )
            btn = gr.Button("🚀 Run Live Comparison", variant="primary", size="lg")
        with gr.Column(scale=2):
            max_tok_slider = gr.Slider(minimum=32, maximum=512, value=250, step=10, label="Max New Tokens")
            temp_slider = gr.Slider(minimum=0.0, maximum=1.0, value=0.2, step=0.05, label="Temperature")
            top_p_slider = gr.Slider(minimum=0.1, maximum=1.0, value=0.9, step=0.05, label="Top-p")
            
    with gr.Row():
        with gr.Column():
            base_out = gr.Textbox(label="🔴 1. Base Model (SmolLM2-1.7B Raw)", lines=12, elem_classes="model-box")
        with gr.Column():
            sft_out = gr.Textbox(label="🟡 2. SFT Model (SmolTalk Aligned)", lines=12, elem_classes="model-box")
        with gr.Column():
            dpo_out = gr.Textbox(label="🟢 3. DPO Model (UltraFeedback Preferred)", lines=12, elem_classes="model-box")
            
    gr.Examples(
        examples=[
            ["What is the difference between a software library and a framework? Explain with a simple analogy.", 250, 0.2, 0.9],
            ["A store has 120 apples. They sell 45 in the morning and 30 in the afternoon. How many are left?", 200, 0.0, 1.0],
            ["If I have $50 and spend 40% of it, how much do I have left? Show calculation.", 200, 0.0, 1.0],
            ["Write a polite email to a customer explaining that their shipment will be delayed by 2 days due to weather conditions.", 250, 0.3, 0.9],
            ["Explain in simple terms why the sky is blue.", 250, 0.2, 0.9],
            ["What are three practical tips for staying focused while studying?", 250, 0.3, 0.9],
        ],
        inputs=[prompt_box, max_tok_slider, temp_slider, top_p_slider]
    )
    
    btn.click(
        fn=arena_compare,
        inputs=[prompt_box, max_tok_slider, temp_slider, top_p_slider],
        outputs=[base_out, sft_out, dpo_out]
    )

demo.launch(share=True, inline=True)
